# Demo: Introduction to LangChain as an Orchestration Framework
### Module 5, Topic 4 — RAG from Scratch

**What you'll see in this notebook:**
1. Rebuild the exact same loading → chunking → embedding → storing → retrieving pipeline from Topics 2–3
2. This time, using LangChain's own components instead of hand-written functions
3. Run the same test questions and compare the results side by side

Nothing conceptually new happens here — every step maps directly onto something already built by hand. What changes is *how much code it takes*.


## Step 0 — Install LangChain and Its Voyage AI Integration

We're reusing the same Voyage AI account from Topic 3 — `VOYAGE_API_KEY` should already be set.

In [ ]:
!pip install langchain langchain-community langchain-text-splitters langchain-voyageai faiss-cpu --quiet

## Step 1 — The Same Document, One More Time

This is the identical Naija One Bank policy document used in Topics 2 and 3.

In [ ]:
document = """Naija One Bank — Flexi Save Account Policy (Effective 2026)

The Flexi Save account is Naija One Bank's flagship savings product for individual customers. It is designed for customers who want easy access to their funds while still earning competitive interest. The account has no monthly maintenance fee as long as the minimum balance is maintained.

Interest is calculated daily and credited monthly at a rate of 4.2% per annum. The minimum opening balance required to activate the account is NGN 5,000. To continue earning interest, customers must maintain a minimum balance of NGN 1,000 at all times.

Customers are permitted 3 free withdrawals per month. A fee of NGN 500 applies to each withdrawal beyond this limit. Withdrawals can be made via the mobile app, at any branch, or through an ATM using the Flexi Save debit card.

Accounts that fall below the minimum balance for more than 60 consecutive days will be automatically converted to a Basic Save account, which does not earn interest. Customers can reactivate Flexi Save status by restoring the minimum balance."""

print(f"Document length: {len(document)} characters")

## Step 2 — Split It With LangChain's Text Splitter

Topic 2 built `sentence_aware_chunks()` by hand. Here, LangChain's `RecursiveCharacterTextSplitter` does the equivalent job — it tries to split on paragraph breaks first, then sentences, then words, only falling back to a harder cut if it has to.

We'll use the same target size (140) and overlap (40) as Topic 2, so the comparison is fair.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=140, chunk_overlap=40)
chunks = splitter.split_text(document)

print(f"Number of chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

## Step 3 — Compare to Topic 2's Output

Look for the same signal we checked for by hand in Topic 2: does the "4.2% per annum" sentence stay whole in a single chunk?

You may notice LangChain's splitter doesn't always produce the exact same cut points as our hand-written sentence splitter — it uses a slightly different strategy (paragraph and word boundaries as fallbacks, not strict sentence detection). That's expected: **same goal, different implementation.** The point isn't identical output — it's that a tested, configurable component now does in one line what took a custom function to do in Topic 2.

## Step 4 — Set Up the Embeddings and Vector Store

This step replaces the entire `knowledge_base` list and manual `vo.embed()` calls from Topic 3. `FAISS.from_texts()` embeds every chunk and stores the vectors, in one call.

In [ ]:
from langchain_voyageai import VoyageAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings_model = VoyageAIEmbeddings(model="voyage-4")

vector_store = FAISS.from_texts(chunks, embeddings_model)

print("Vector store built from", len(chunks), "chunks.")

## Step 5 — Turn It Into a Retriever

One line replaces the entire hand-written `retrieve()` function from Topic 3.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("Retriever ready.")

## Step 6 — Ask the Same Question as Topic 3

This is the exact question the from-scratch retriever handled — "take out" and "charged" instead of "withdrawal" and "fee".

In [ ]:
question = "How much can I take out of my account each month before I get charged?"

results = retriever.invoke(question)

for i, doc in enumerate(results):
    print(f"#{i+1}")
    print(doc.page_content)
    print()

## Step 7 — Compare the Two Retrievers

Put this next to Topic 3's Step 7 output. Both should surface the withdrawal-limit chunk at or near the top, for the same underlying reason: the embedding model is matching on meaning, not exact wording — Voyage AI is doing the same job in both notebooks, just called through a different interface.

**What's actually different between the two versions?**

| | Topic 3 (from scratch) | This notebook (LangChain) |
|--|------------------------|------------------------------|
| Chunking | Custom `sentence_aware_chunks()` | `RecursiveCharacterTextSplitter` |
| Storing embeddings | A plain Python list of tuples | `FAISS` vector store |
| Similarity search | Hand-written `cosine_similarity()` + sort | Built into `.as_retriever()` |
| Lines of retrieval logic | ~15 | ~5 |
| What we understand | Exactly what happens at every step | The same steps, running inside tested components |

## Step 8 — Try the Second Question Too

The same "two months" question from Topic 3 — testing whether this retriever also correctly finds the account-conversion chunk.

In [ ]:
new_question = "What happens if my balance drops too low for two months?"

results = retriever.invoke(new_question)

for i, doc in enumerate(results):
    print(f"#{i+1}")
    print(doc.page_content)
    print()

## What's Next

We now have two working retrievers — one built by hand in Topic 3, one built with LangChain here — solving the exact same problem. In Topic 5, either retriever gets connected to an LLM call to complete the RAG pipeline: retrieve the chunks, then generate an answer grounded in them.